In [ ]:
sys.path.insert(0, os.path.join('..'))

import glob
import ipywidgets
import os
import sys

from functools import reduce

import geopandas as gpd
import cartopy.io.shapereader as shpreader
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from utils.config import get_results_path

In [ ]:


PATH_RETURNLVLS = get_results_path()

In [ ]:
## Select the file for the desired return level
FILES = glob.glob(os.path.join(PATH_RETURNLVLS, "*.csv"))
FILE_NAMES = sorted(
    [os.path.basename(x) for x in glob.glob(os.path.join(PATH_RETURNLVLS, "*.csv"))]
)

csv_files = ipywidgets.SelectMultiple(
    options=FILE_NAMES,
    value=FILE_NAMES,
    description="Choose return period:",
    disabled=False,
)
csv_files

In [ ]:
df_temp = pd.read_csv(os.path.join(PATH_RETURNLVLS, csv_files.value[0]))

avail_cols = sorted(list(set(df_temp.columns) - set(["lon", "lat"])))

sel_cols = ipywidgets.SelectMultiple(
    options=avail_cols, value=avail_cols, description="Choose columns:", disabled=False
)

sel_cols

In [ ]:
df_append = []

# append all files together
for file in csv_files.value:
    df_temp = pd.read_csv(os.path.join(PATH_RETURNLVLS, file))
    df_temp = df_temp[["lon", "lat"] + list(sel_cols.value)]

    for col in sel_cols.value:
        df_temp[col]
        df_temp.rename(
            columns={col: f"{os.path.splitext(file)[0]}_{col}"}, inplace=True
        )

    df_append.append(df_temp)

In [ ]:
return_data_original = reduce(
    lambda x, y: pd.merge(x, y, on=["lon", "lat"], how="outer"), df_append
)

## Get some inner Austria statistics about the return levels

In [ ]:
# Get Natural Earth countries shapefile (comes with cartopy)
shpfilename = shpreader.natural_earth(resolution='10m',
                                       category='cultural',
                                       name='admin_0_countries')

# Read with geopandas and filter for Austria
world = gpd.read_file(shpfilename)
austria = world[world['ADMIN'] == 'Austria']

print(f"Austria boundary loaded successfully from cartopy's Natural Earth data!")
print(f"Austria bounds: {austria.total_bounds}")  # [minx, miny, maxx, maxy]


In [ ]:
# Create a 1x1km grid of points within Austria
# This is solely used to know the number of grid points within Austria
import numpy as np
from shapely.geometry import Point

# Project Austria to a metric coordinate system (UTM 33N covers most of Austria)
austria_utm = austria.to_crs('EPSG:32633')  # UTM zone 33N

# Get bounds in meters
minx, miny, maxx, maxy = austria_utm.total_bounds

# Create 1km grid (1000 meters)
grid_resolution = 1000  # meters
x_coords = np.arange(minx, maxx, grid_resolution)
y_coords = np.arange(miny, maxy, grid_resolution)

# Create all grid points
xx, yy = np.meshgrid(x_coords, y_coords)
grid_points_x = xx.flatten()
grid_points_y = yy.flatten()

print(f"Total grid points in bounding box: {len(grid_points_x):,}")

# Create GeoDataFrame of grid points in UTM
grid_geometry = [Point(x, y) for x, y in zip(grid_points_x, grid_points_y)]
grid_gdf = gpd.GeoDataFrame(geometry=grid_geometry, crs='EPSG:32633')

# Filter to only points within Austria
grid_austria = gpd.sjoin(grid_gdf, austria_utm, how='inner', predicate='within')

# Convert back to lat/lon (EPSG:4326)
grid_austria = grid_austria.to_crs('EPSG:4326')

# Extract lon, lat coordinates
grid_austria['lon'] = grid_austria.geometry.x
grid_austria['lat'] = grid_austria.geometry.y

# Keep only relevant columns
grid_austria = grid_austria[['lon', 'lat', 'geometry']].reset_index(drop=True)

nr_gridpoints_austria = len(grid_austria)
print(f"Grid points within Austria: {nr_gridpoints_austria:,}")
print(f"Percentage within Austria: {nr_gridpoints_austria / len(grid_points_x) * 100:.1f}%")
print(f"\nLongitude range: {grid_austria['lon'].min():.4f} to {grid_austria['lon'].max():.4f}")
print(f"Latitude range: {grid_austria['lat'].min():.4f} to {grid_austria['lat'].max():.4f}")

In [ ]:
# Filter return_data to only include coordinates within Austria
from shapely.geometry import Point

# Create a GeoDataFrame from return_data
geometry = [Point(lon, lat) for lon, lat in zip(return_data_original['lon'], return_data_original['lat'])]
gdf_points = gpd.GeoDataFrame(return_data_original, geometry=geometry, crs='EPSG:4326')

# Ensure austria has the same CRS
austria = austria.to_crs('EPSG:4326')

# Spatial join to keep only points within Austria
gdf_austria = gpd.sjoin(gdf_points, austria, how='inner', predicate='within')

# Drop the extra columns added by the spatial join
cols_to_drop = [col for col in gdf_austria.columns if col.startswith('index_')]
gdf_austria = gdf_austria.drop(columns=cols_to_drop + ['geometry'])

# Update return_data with filtered data
return_data = gdf_austria.reset_index(drop=True)

print(f"Original data points: {len(return_data_original)}")
print(f"Points within Austria: {len(return_data)}")
print(f"Percentage within Austria: {len(return_data) / len(return_data_original) * 100:.2f}%")

In [ ]:
return_data.describe()

In [ ]:
nr_most_severe = len(return_data.query("bootstrap_results_30_median >= 50")) / nr_gridpoints_austria

print(f"Percentage of gridpoints within Austria belonging to the most severe class (30y): {nr_most_severe:.2%}")

In [ ]:
for var in sel_cols.value:
    # Prepare data similar to plotting code
    plot_data = return_data[[colname for colname in return_data.columns if colname.endswith(var)]].copy() / 10
    plot_data.columns = ['10 years', '20 years', '30 years']

    print(f"\nStatistics for {var}:")
    print("-" * 60)
    for period in ['10 years', '20 years', '30 years']:
        median = plot_data[period].median()
        q25 = plot_data[period].quantile(0.25)
        q75 = plot_data[period].quantile(0.75)
        iqr = q75 - q25
        iqr_1_5 = 1.5 * iqr
        lower_bound = q25 - iqr_1_5
        upper_bound = q75 + iqr_1_5
        print()
        print(f"{period}:")
        print(f"  25th percentile:     {q25:.2f} cm")
        print(f"  Median:              {median:.2f} cm")
        print(f"  75th percentile:     {q75:.2f} cm")
        print(f"  IQR:                 {iqr:.2f} cm")
        print(f"  1.5 × IQR:           {iqr_1_5:.2f} cm")
        print(f"  Median - 1.5 × IQR:  {lower_bound:.2f} cm")
        print(f"  Median + 1.5 × IQR:  {upper_bound:.2f} cm")

In [ ]:
for boxenplot in [False, True]:
    filename = f"return_levels_trend{'_boxenplot' if boxenplot else '_boxplot'}"

    for var in sel_cols.value:
        is_conf_band = "conf" in var

        # Prepare data for boxplot
        plot_data = return_data[[colname for colname in return_data.columns if colname.endswith(var)]].copy() / 10

        # Rename columns for display
        plot_data.columns = ['10 years', '20 years', '30 years']

        # Reshape data for seaborn boxplot
        plot_data_melted = plot_data.melt(var_name='Period', value_name='Median Value')

        # Create boxplot
        plt.rcParams.update({'font.size': 36})
        ax = plt.figure(figsize=(15, 10))

        kwargs = {
            "data": plot_data_melted,
            "x": 'Period',
            "y": 'Median Value'
        }

        if boxenplot:
            sns.boxenplot(**kwargs)
        else:
            sns.boxplot(**kwargs)

        if not is_conf_band:
            plt.axhline(y=5, linestyle='--', color='lightgreen', alpha=0.7)

        title = 'Confidence Interval' if is_conf_band else 'Return levels'
        title = '98% ' + title if '98p' in var else title

        plt.title(title)
        plt.ylabel('[cm]' if is_conf_band else 'hailstone size [cm]')
        plt.xlabel('period')
        plt.tight_layout()
        plt.savefig(os.path.join(PATH_RETURNLVLS, f"{filename}_{var}.png"), bbox_inches="tight")
        plt.savefig(os.path.join(PATH_RETURNLVLS, f"{filename}_{var}.pdf"), bbox_inches="tight")

        plt.close()